In [1]:
from pathlib import Path
import duckdb

In [2]:
data_dir = Path("/media/datasets/smol-smoltalk/data")
num_samples = 10

In [3]:
parquet_files = list(data_dir.rglob("*.parquet"))
glob_pattern = str(data_dir / "**" / "*.parquet")
print(glob_pattern)
parquet_files

/media/datasets/smol-smoltalk/data/**/*.parquet


[PosixPath('/media/datasets/smol-smoltalk/data/train-00000-of-00004.parquet'),
 PosixPath('/media/datasets/smol-smoltalk/data/train-00002-of-00004.parquet'),
 PosixPath('/media/datasets/smol-smoltalk/data/train-00003-of-00004.parquet'),
 PosixPath('/media/datasets/smol-smoltalk/data/train-00001-of-00004.parquet'),
 PosixPath('/media/datasets/smol-smoltalk/data/test-00000-of-00001.parquet')]

In [4]:
con = duckdb.connect()

print("=" * 60)
print("DATASET SCHEMA & TYPES")
print("=" * 60)
schema_df = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{glob_pattern}')"
).pl()
print(schema_df[["column_name", "column_type"]])

DATASET SCHEMA & TYPES
shape: (2, 2)
┌─────────────┬─────────────────────────────────┐
│ column_name ┆ column_type                     │
│ ---         ┆ ---                             │
│ str         ┆ str                             │
╞═════════════╪═════════════════════════════════╡
│ messages    ┆ STRUCT("content" VARCHAR, "rol… │
│ source      ┆ VARCHAR                         │
└─────────────┴─────────────────────────────────┘


In [6]:
print("\n" + "=" * 60)
print("GLOBAL DATASET AGGREGATES")
print("=" * 60)
stats_query = f"""
    SELECT 
        COUNT(*) AS total_documents,
        ROUND(AVG(LENGTH(messages)), 2) AS avg_char_len,
        MEDIAN(LENGTH(messages)) AS median_char_len,
        MIN(LENGTH(messages)) AS min_char_len,
        MAX(LENGTH(messages)) AS max_char_len,
        -- Rough word estimate (~5 chars per word)
        ROUND(SUM(LENGTH(messages)) / 5.0 / 1e9, 3) AS approx_billion_words
    FROM read_parquet('{glob_pattern}')
"""
stats_df = con.sql(stats_query).pl()
print(stats_df)


GLOBAL DATASET AGGREGATES
shape: (1, 6)
┌─────────────────┬──────────────┬─────────────────┬──────────────┬──────────────┬─────────────────┐
│ total_documents ┆ avg_char_len ┆ median_char_len ┆ min_char_len ┆ max_char_len ┆ approx_billion_ │
│ ---             ┆ ---          ┆ ---             ┆ ---          ┆ ---          ┆ words           │
│ i64             ┆ f64          ┆ f64             ┆ i64          ┆ i64          ┆ ---             │
│                 ┆              ┆                 ┆              ┆              ┆ f64             │
╞═════════════════╪══════════════╪═════════════════╪══════════════╪══════════════╪═════════════════╡
│ 484570          ┆ 4.64         ┆ 6.0             ┆ 2            ┆ 58           ┆ 0.0             │
└─────────────────┴──────────────┴─────────────────┴──────────────┴──────────────┴─────────────────┘


In [13]:
print("\n" + "=" * 60)
print(f"SAMPLE DOCUMENTS (Showing {num_samples})")
print("=" * 60)
samples_query = f"""
    SELECT messages, source
    FROM read_parquet('{glob_pattern}')
    LIMIT {num_samples}
"""
samples = con.sql(samples_query).pl().to_dicts()

for idx, sample in enumerate(samples, 1):
    print(f"\n--- [Sample {idx}] ---")
    print(f"source: {sample['source']}")
    print("-" * 40)
    for msg in sample['messages']:
        print(f'[{msg["role"]}] {msg["content"]}')
    print("-" * 40)


SAMPLE DOCUMENTS (Showing 10)

--- [Sample 1] ---
source: smol-summarize-20k
----------------------------------------
[system] Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns.
[user] Uruguay hero Fernando Muslera was delighted after his team stunned hosts Argentina in a penalty shootout to reach the semifinals of the Copa America tournament, despite having a player sent off. The goalkeeper saved a spot-kick by striker Carlos Tevez to earn "La Celeste" a clash with Peru, who beat Colombia 2-0 after extra-time in Cordoba earlier on Saturday. The Uruguayans, who reached the last four at the 2010 World Cup, will be seeking to make the Copa final for the first time since 1999 when they line up in La Plata on Tuesday. The upset victory against close neighbors Argentina came exactly 61 years after Uruguay won the World Cup by stunning Brazil in Rio de Janeiro's Maracana stadi